# 🧠 Tasks 2–4: Modelling & Training — Flickr8k

This notebook trains all model variants and saves all results to disk.
**No visualizations here** — run `03_model_analysis.ipynb` locally for all plots.

**Saves to disk:**
- `saved_models/baseline_gru.pth`
- `saved_models/baseline_lstm.pth`
- `saved_models/gru_attn.pth`
- `saved_models/lstm_attn.pth`
- `artifacts/training_results.pkl` — all histories + ablation results

In [1]:
import os, warnings, pickle
warnings.filterwarnings('ignore')
import torch

from models import (
    load_artifacts, build_dataloaders, get_device,
    build_model, train_model, save_checkpoint,
)

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch 2.8.0+cu128
CUDA available: True
GPU: NVIDIA H100 80GB HBM3


## 0. Setup — Load Artifacts & Hyperparameters

In [2]:
# ── Load EDA artifacts ──
artifacts = load_artifacts('artifacts/eda_artifacts.pkl')
VOCAB_SIZE = artifacts['vocab_size']
MAX_SEQ    = artifacts['max_seq_len']

device = get_device()
print(f"Vocab size : {VOCAB_SIZE}")
print(f"Max seq len: {MAX_SEQ}")

# ── Hyperparameters ──
EMBED_DIM  = 256
UNITS      = 256
DROPOUT    = 0.4
BATCH_SIZE = 512
EPOCHS     = 15
LR         = 1e-3
USE_AMP    = True   # bfloat16 on H100

# ── Directories ──
os.makedirs('saved_models', exist_ok=True)
os.makedirs('artifacts',    exist_ok=True)

# ── Build data loaders ──
train_loader, val_loader, test_loader = build_dataloaders(
    artifacts, batch_size=BATCH_SIZE, num_workers=4
)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

🖥️  GPU 0: NVIDIA H100 80GB HBM3
Vocab size : 2988
Max seq len: 26
Train batches: 64 | Val batches: 8


## W&B (optional)

In [3]:
USE_WANDB = True  # Set True to enable Weights & Biases tracking

if USE_WANDB:
    import wandb
    wandb.login()
    print('W&B ready.')
else:
    print('W&B disabled.')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /teamspace/studios/this_studio/.netrc.


wandb: Currently logged in as: hamzaheshamhendy (hamzaheshamhendy-infor) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B ready.


---
## Tasks 2 & 3: Baseline Models

### Architecture
```
Image → ResNet50 (frozen) → Linear → h₀
Caption[:-1] → Embedding → GRU/LSTM(h₀) → Dense → Predicted tokens
```
Trained with **teacher forcing**: at each timestep, the ground-truth previous token is fed as input.

In [4]:
# ── Baseline GRU ──
print('='*60)
print('Training: Baseline GRU')
print('='*60)

if USE_WANDB:
    import wandb
    wandb.init(project='flickr8k-captioning', name='baseline_gru', reinit=True,
               config={'model': 'gru', 'embed': EMBED_DIM, 'units': UNITS,
                       'dropout': DROPOUT, 'batch': BATCH_SIZE, 'lr': LR})

gru_model = build_model('gru', VOCAB_SIZE, EMBED_DIM, UNITS, DROPOUT).to(device)
gru_hist  = train_model(gru_model, train_loader, val_loader, device,
                        epochs=EPOCHS, lr=LR, use_amp=USE_AMP,
                        use_wandb=USE_WANDB, run_name='baseline_gru')

save_checkpoint(gru_model, gru_hist,
    {'model': 'gru', 'embed': EMBED_DIM, 'units': UNITS, 'dropout': DROPOUT},
    'saved_models/baseline_gru.pth')

if USE_WANDB: wandb.finish()
print(f"GRU done. Best val loss: {gru_hist['best_val_loss']:.4f} | Time: {gru_hist['elapsed_s']:.1f}s")

Training: Baseline GRU


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Epoch 1/15 | Train 5.1159 | Val 4.0566 | LR 1.00e-03
   ✨ New best val loss: 4.0566 — weights saved


Epoch 2/15 | Train 3.8628 | Val 3.5927 | LR 1.00e-03
   ✨ New best val loss: 3.5927 — weights saved


Epoch 3/15 | Train 3.5082 | Val 3.3513 | LR 1.00e-03
   ✨ New best val loss: 3.3513 — weights saved


...

Epoch 13/15 | Train 2.4771 | Val 2.8437 | LR 1.00e-03


Epoch 14/15 | Train 2.4282 | Val 2.8385 | LR 1.00e-03
   ✨ New best val loss: 2.8385 — weights saved


Epoch 15/15 | Train 2.3806 | Val 2.8573 | LR 1.00e-03
💾 Saved → saved_models/baseline_gru.pth


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,█▅▄▃▃▃▂▂▂▂▂▁▁▁▁
val_loss,█▅▄▃▂▂▂▁▁▁▁▁▁▁▁
epoch,15
lr,0.001
train_loss,2.38056
val_loss,2.85733


GRU done. Best val loss: 2.8385 | Time: 607.3s


In [5]:
# ── Baseline LSTM ──
print('='*60)
print('Training: Baseline LSTM')
print('='*60)

if USE_WANDB:
    import wandb
    wandb.init(project='flickr8k-captioning', name='baseline_lstm', reinit=True,
               config={'model': 'lstm', 'embed': EMBED_DIM, 'units': UNITS,
                       'dropout': DROPOUT, 'batch': BATCH_SIZE, 'lr': LR})

lstm_model = build_model('lstm', VOCAB_SIZE, EMBED_DIM, UNITS, DROPOUT).to(device)
lstm_hist  = train_model(lstm_model, train_loader, val_loader, device,
                         epochs=EPOCHS, lr=LR, use_amp=USE_AMP,
                         use_wandb=USE_WANDB, run_name='baseline_lstm')

save_checkpoint(lstm_model, lstm_hist,
    {'model': 'lstm', 'embed': EMBED_DIM, 'units': UNITS, 'dropout': DROPOUT},
    'saved_models/baseline_lstm.pth')

if USE_WANDB: wandb.finish()
print(f"LSTM done. Best val loss: {lstm_hist['best_val_loss']:.4f} | Time: {lstm_hist['elapsed_s']:.1f}s")

Training: Baseline LSTM


Epoch 1/15 | Train 5.2759 | Val 4.2641 | LR 1.00e-03
   ✨ New best val loss: 4.2641 — weights saved


Epoch 2/15 | Train 4.0700 | Val 3.8123 | LR 1.00e-03
   ✨ New best val loss: 3.8123 — weights saved


Epoch 3/15 | Train 3.7236 | Val 3.5655 | LR 1.00e-03
   ✨ New best val loss: 3.5655 — weights saved


Epoch 4/15 | Train 3.5085 | Val 3.4122 | LR 1.00e-03
   ✨ New best val loss: 3.4122 — weights saved


...

Epoch 13/15 | Train 2.7126 | Val 2.9484 | LR 1.00e-03
   ✨ New best val loss: 2.9484 — weights saved


Epoch 14/15 | Train 2.6622 | Val 2.9366 | LR 1.00e-03
   ✨ New best val loss: 2.9366 — weights saved


Epoch 15/15 | Train 2.6114 | Val 2.9302 | LR 1.00e-03
   ✨ New best val loss: 2.9302 — weights saved
💾 Saved → saved_models/baseline_lstm.pth


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,█▅▄▃▃▃▂▂▂▂▂▁▁▁▁
val_loss,█▆▄▄▃▂▂▂▂▁▁▁▁▁▁
epoch,15
lr,0.001
train_loss,2.61144
val_loss,2.93019


LSTM done. Best val loss: 2.9302 | Time: 604.8s


---
## Task 4: Enhanced Models — Bahdanau Attention

### Architecture
```
Image → ResNet50 (frozen) → Linear → (B, 49, D)  [spatial features]
For each timestep t:
    context = BahdanauAttention(features, h_{t-1})
    h_t     = GRUCell/LSTMCell([embed(word_t) || context], h_{t-1})
    out_t   = Dense(h_t)
```

In [6]:
# ── GRU + Bahdanau Attention ──
print('='*60)
print('Training: GRU + Bahdanau Attention')
print('='*60)

if USE_WANDB:
    import wandb
    wandb.init(project='flickr8k-captioning', name='gru_attention', reinit=True,
               config={'model': 'gru_attn', 'embed': EMBED_DIM, 'units': UNITS,
                       'dropout': DROPOUT, 'batch': BATCH_SIZE, 'lr': LR})

gru_attn_model = build_model('gru_attn', VOCAB_SIZE, EMBED_DIM, UNITS, DROPOUT).to(device)
gru_attn_hist  = train_model(gru_attn_model, train_loader, val_loader, device,
                             epochs=EPOCHS, lr=LR, use_amp=USE_AMP,
                             use_wandb=USE_WANDB, run_name='gru_attention')

save_checkpoint(gru_attn_model, gru_attn_hist,
    {'model': 'gru_attn', 'embed': EMBED_DIM, 'units': UNITS, 'dropout': DROPOUT},
    'saved_models/gru_attn.pth')

if USE_WANDB: wandb.finish()
print(f"GRU+Attn done. Best val loss: {gru_attn_hist['best_val_loss']:.4f} | Time: {gru_attn_hist['elapsed_s']:.1f}s")

Training: GRU + Bahdanau Attention


Epoch 1/15 | Train 5.1877 | Val 4.3070 | LR 1.00e-03
   ✨ New best val loss: 4.3070 — weights saved


Epoch 2/15 | Train 4.0615 | Val 3.7104 | LR 1.00e-03
   ✨ New best val loss: 3.7104 — weights saved


Epoch 3/15 | Train 3.6074 | Val 3.4066 | LR 1.00e-03
   ✨ New best val loss: 3.4066 — weights saved


Epoch 4/15 | Train 3.3303 | Val 3.2170 | LR 1.00e-03
   ✨ New best val loss: 3.2170 — weights saved


...

Epoch 13/15 | Train 2.3817 | Val 2.9202 | LR 5.00e-04


Epoch 14/15 | Train 2.2799 | Val 2.9345 | LR 5.00e-04


Epoch 15/15 | Train 2.2326 | Val 2.9550 | LR 5.00e-04
💾 Saved → saved_models/gru_attn.pth


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
lr,████████████▁▁▁
train_loss,█▅▄▄▃▃▃▂▂▂▂▂▁▁▁
val_loss,█▅▄▃▂▂▁▁▁▁▁▁▁▁▁
epoch,15
lr,0.0005
train_loss,2.23257
val_loss,2.95497


GRU+Attn done. Best val loss: 2.9037 | Time: 607.9s


In [7]:
# ── LSTM + Bahdanau Attention ──
print('='*60)
print('Training: LSTM + Bahdanau Attention')
print('='*60)

if USE_WANDB:
    import wandb
    wandb.init(project='flickr8k-captioning', name='lstm_attention', reinit=True,
               config={'model': 'lstm_attn', 'embed': EMBED_DIM, 'units': UNITS,
                       'dropout': DROPOUT, 'batch': BATCH_SIZE, 'lr': LR})

lstm_attn_model = build_model('lstm_attn', VOCAB_SIZE, EMBED_DIM, UNITS, DROPOUT).to(device)
lstm_attn_hist  = train_model(lstm_attn_model, train_loader, val_loader, device,
                              epochs=EPOCHS, lr=LR, use_amp=USE_AMP,
                              use_wandb=USE_WANDB, run_name='lstm_attention')

save_checkpoint(lstm_attn_model, lstm_attn_hist,
    {'model': 'lstm_attn', 'embed': EMBED_DIM, 'units': UNITS, 'dropout': DROPOUT},
    'saved_models/lstm_attn.pth')

if USE_WANDB: wandb.finish()
print(f"LSTM+Attn done. Best val loss: {lstm_attn_hist['best_val_loss']:.4f} | Time: {lstm_attn_hist['elapsed_s']:.1f}s")

Training: LSTM + Bahdanau Attention


Epoch 1/15 | Train 5.4716 | Val 4.8288 | LR 1.00e-03
   ✨ New best val loss: 4.8288 — weights saved


Epoch 2/15 | Train 4.5964 | Val 4.2562 | LR 1.00e-03
   ✨ New best val loss: 4.2562 — weights saved


Epoch 3/15 | Train 4.1255 | Val 3.9171 | LR 1.00e-03
   ✨ New best val loss: 3.9171 — weights saved


Epoch 4/15 | Train 3.8416 | Val 3.6838 | LR 1.00e-03
   ✨ New best val loss: 3.6838 — weights saved


...

Epoch 13/15 | Train 2.6721 | Val 2.9938 | LR 1.00e-03
   ✨ New best val loss: 2.9938 — weights saved


Epoch 14/15 | Train 2.6018 | Val 2.9894 | LR 1.00e-03
   ✨ New best val loss: 2.9894 — weights saved


Epoch 15/15 | Train 2.5373 | Val 2.9943 | LR 1.00e-03
💾 Saved → saved_models/lstm_attn.pth


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,█▆▅▄▄▃▃▂▂▂▂▁▁▁▁
val_loss,█▆▅▄▃▂▂▂▁▁▁▁▁▁▁
epoch,15
lr,0.001
train_loss,2.53735
val_loss,2.99426


LSTM+Attn done. Best val loss: 2.9894 | Time: 609.7s


---
## Task 4: Embedding Size Ablation

Compare `embed_dim = 128 / 256 / 512` on the GRU baseline.
All other hyperparameters (units, dropout, lr, epochs) are fixed — one variable changes at a time.

Note: `embed_dim=256` is re-trained here for fair comparison (same number of ablation epochs).

In [8]:
ABLATION_EPOCHS = 5          # shorter run — enough to show the trend
EMBED_SIZES     = [128, 256, 512]
ablation_results = {}        # embed_dim → history dict

for emb in EMBED_SIZES:
    print(f"\n{'='*50}")
    print(f"Embedding size = {emb}")
    print(f"{'='*50}")

    m = build_model('gru', VOCAB_SIZE, embed_dim=emb, units=UNITS, dropout=DROPOUT).to(device)

    hist = train_model(m, train_loader, val_loader, device,
                       epochs=ABLATION_EPOCHS, lr=LR,
                       use_amp=USE_AMP, use_wandb=False,
                       run_name=f'gru_embed{emb}')

    ablation_results[emb] = hist
    print(f"  embed={emb} | Best val loss: {hist['best_val_loss']:.4f} | Time: {hist['elapsed_s']:.1f}s")

print("\nAblation complete.")


Embedding size = 128


Epoch 1/5 | Train 5.2520 | Val 4.2564 | LR 1.00e-03
   ✨ New best val loss: 4.2564 — weights saved


Epoch 2/5 | Train 4.0457 | Val 3.7443 | LR 1.00e-03
   ✨ New best val loss: 3.7443 — weights saved


Epoch 3/5 | Train 3.6457 | Val 3.4496 | LR 1.00e-03
   ✨ New best val loss: 3.4496 — weights saved


Epoch 4/5 | Train 3.3769 | Val 3.2502 | LR 1.00e-03
   ✨ New best val loss: 3.2502 — weights saved


...

Epoch 1/5 | Train 4.9835 | Val 3.9237 | LR 1.00e-03
   ✨ New best val loss: 3.9237 — weights saved


Epoch 2/5 | Train 3.7516 | Val 3.4973 | LR 1.00e-03
   ✨ New best val loss: 3.4973 — weights saved


Epoch 3/5 | Train 3.4186 | Val 3.2725 | LR 1.00e-03
   ✨ New best val loss: 3.2725 — weights saved


Epoch 4/5 | Train 3.2104 | Val 3.1290 | LR 1.00e-03
   ✨ New best val loss: 3.1290 — weights saved


Epoch 5/5 | Train 3.0550 | Val 3.0304 | LR 1.00e-03
   ✨ New best val loss: 3.0304 — weights saved
  embed=512 | Best val loss: 3.0304 | Time: 196.2s

Ablation complete.


---
## Save All Training Results

Saves a single `training_results.pkl` containing all histories and the ablation results.
The analysis notebook loads this file to produce all plots.

In [9]:
import pickle

training_results = {
    # Per-model training histories
    'histories': {
        'Baseline GRU':     gru_hist,
        'Baseline LSTM':    lstm_hist,
        'GRU + Attention':  gru_attn_hist,
        'LSTM + Attention': lstm_attn_hist,
    },
    # Embedding size ablation results
    'ablation': ablation_results,       # {embed_dim: history_dict}
    'ablation_epochs': ABLATION_EPOCHS,
    # Hyperparameter record
    'hyperparameters': {
        'embed_dim':  EMBED_DIM,
        'units':      UNITS,
        'dropout':    DROPOUT,
        'batch_size': BATCH_SIZE,
        'epochs':     EPOCHS,
        'lr':         LR,
    },
}

save_path = 'artifacts/training_results.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(training_results, f)

print(f"✅ Saved training results → {save_path}")
print(f"\n{'Model':<25} {'Best Val Loss':>14} {'Time (s)':>10}")
print('-' * 52)
for name, hist in training_results['histories'].items():
    print(f"{name:<25} {hist['best_val_loss']:>14.4f} {hist['elapsed_s']:>10.1f}")
print(f"\nAblation (embed dim vs best val loss):")
for emb, hist in ablation_results.items():
    print(f"  embed={emb:>3}: {hist['best_val_loss']:.4f}")

✅ Saved training results → artifacts/training_results.pkl

Model                      Best Val Loss   Time (s)
----------------------------------------------------
Baseline GRU                      2.8385      607.3
Baseline LSTM                     2.9302      604.8
GRU + Attention                   2.9037      607.9
LSTM + Attention                  2.9894      609.7

Ablation (embed dim vs best val loss):
  embed=128: 3.1352
  embed=256: 3.0994
  embed=512: 3.0304
